# Binary Classification Assignment Pipeline
### Decision Tree Classification vs. XGBoost Classification
**Dataset:** `BC-BrainCancer.csv`

This notebook mirrors the regression pipeline exactly (sequential, non-shuffled
train/validation/test split; Decision Tree and XGBoost models; validation-only
hyperparameter tuning; manually implemented evaluation metrics with no
`sklearn.metrics` calls; final results comparison) but applies it to a
**binary classification** task: predicting patient `status` (0 = alive/censored,
1 = event) from clinical/demographic features.

## Section 1: Import Libraries

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Section 9 (defined early so it can be reused throughout)
### Manual Classification Metrics
No library metric function (e.g. `sklearn.metrics.accuracy_score`,
`sklearn.metrics.precision_score`, etc.) is used anywhere in this notebook.
All metrics are implemented directly with NumPy from the confusion-matrix counts:

- True Positive (TP): predicted 1, actual 1
- True Negative (TN): predicted 0, actual 0
- False Positive (FP): predicted 1, actual 0
- False Negative (FN): predicted 0, actual 1

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} \qquad
\text{Precision} = \frac{TP}{TP + FP} \qquad
\text{Recall} = \frac{TP}{TP + FN} \qquad
\text{F1} = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

`manual_accuracy` is used as the **primary metric for hyperparameter selection**
on the validation set (the classification analogue of `manual_mse` in the
regression notebook, i.e. the single scalar the validation-based search
maximizes). Precision, Recall, F1, and the confusion matrix are computed
manually as well and reported for the final test-set evaluation.

In [2]:
def manual_confusion_counts(y_true, y_pred):
    """
    Compute TP, TN, FP, FN counts manually with NumPy for binary labels {0, 1}.
    """
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    return tp, tn, fp, fn


def manual_accuracy(y_true, y_pred):
    """
    Accuracy = (TP + TN) / (TP + TN + FP + FN)
    Steps: (1) get confusion counts, (2) sum correct predictions,
    (3) divide by total number of predictions.
    """
    tp, tn, fp, fn = manual_confusion_counts(y_true, y_pred)  # step 1
    correct = tp + tn                                          # step 2
    total = tp + tn + fp + fn
    return correct / total if total > 0 else 0.0               # step 3


def manual_precision(y_true, y_pred):
    """Precision = TP / (TP + FP), with 0.0 if the denominator is 0."""
    tp, tn, fp, fn = manual_confusion_counts(y_true, y_pred)
    denom = tp + fp
    return tp / denom if denom > 0 else 0.0


def manual_recall(y_true, y_pred):
    """Recall = TP / (TP + FN), with 0.0 if the denominator is 0."""
    tp, tn, fp, fn = manual_confusion_counts(y_true, y_pred)
    denom = tp + fn
    return tp / denom if denom > 0 else 0.0


def manual_f1(y_true, y_pred):
    """F1 = 2 * Precision * Recall / (Precision + Recall), with 0.0 if undefined."""
    p = manual_precision(y_true, y_pred)
    r = manual_recall(y_true, y_pred)
    return (2 * p * r / (p + r)) if (p + r) > 0 else 0.0


def manual_classification_report(y_true, y_pred):
    tp, tn, fp, fn = manual_confusion_counts(y_true, y_pred)
    return {
        "accuracy": manual_accuracy(y_true, y_pred),
        "precision": manual_precision(y_true, y_pred),
        "recall": manual_recall(y_true, y_pred),
        "f1": manual_f1(y_true, y_pred),
        "confusion_matrix": {"TP": tp, "TN": tn, "FP": fp, "FN": fn},
    }

## Section 2: Load Dataset

In [3]:
def load_brain_cancer(path):
    df = pd.read_csv(path)
    # Drop the row-index column exported by the source (unnamed first column).
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
    return df, "status"

## Section 3: Data Inspection
`diagnosis` has a single missing value (1 out of 88 rows). Since this is a
tiny dataset and only one row is affected, the row is dropped rather than
imputed, to avoid introducing artificial values into such a small sample.
This is done explicitly and logged below (not silently).

In [4]:
def inspect(df, name):
    report = {
        "dataset": name,
        "n_rows": int(df.shape[0]),
        "n_cols": int(df.shape[1]),
        "columns": list(df.columns),
        "dtypes": {c: str(t) for c, t in df.dtypes.items()},
        "missing_values": {c: int(v) for c, v in df.isna().sum().items()},
    }
    return report


def clean_and_inspect(df, name):
    insp_before = inspect(df, name)
    n_before = len(df)
    df_clean = df.dropna().reset_index(drop=True)
    n_after = len(df_clean)
    print(f"Rows before dropping missing values: {n_before}")
    print(f"Rows after dropping missing values:  {n_after}  (dropped {n_before - n_after})")
    return df_clean, insp_before


def encode_features(df, target):
    """
    One-hot encode categorical predictors; leave the binary target untouched.
    """
    X = df.drop(columns=[target]).copy()
    y = df[target].copy().reset_index(drop=True).astype(int)
    cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    X = X.reset_index(drop=True)
    return X, y

## Section 4: Sequential Train / Validation / Test Split
Identical to the regression notebook: no shuffling, strict sequential
70% / 15% / 15% split.

In [5]:
def sequential_split(X, y):
    n = len(X)
    train_end = int(np.floor(0.70 * n))
    val_end = int(np.floor(0.85 * n))

    X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
    X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
    X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

## Section 5 & 6: Decision Tree Classifier + Hyperparameter Tuning
Same hyperparameter families as the regression notebook, adapted to
classification: `max_depth`, `min_samples_split`, `min_samples_leaf`, and
`criterion` (the classification split-quality functions `gini`, `entropy`,
`log_loss`, in place of `squared_error`/`friedman_mse`). Exhaustive grid
search (160 combinations) is scored on the validation set with
`manual_accuracy`; the highest-accuracy combination is kept.

In [6]:
DT_GRID = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "criterion": ["gini", "entropy", "log_loss"],
}


def tune_decision_tree(X_train, y_train, X_val, y_val, grid=DT_GRID):
    results = []
    best = {"acc": -np.inf, "params": None, "model": None}
    for depth in grid["max_depth"]:
        for mss in grid["min_samples_split"]:
            for msl in grid["min_samples_leaf"]:
                for crit in grid["criterion"]:
                    params = dict(max_depth=depth, min_samples_split=mss,
                                  min_samples_leaf=msl, criterion=crit,
                                  random_state=RANDOM_STATE)
                    model = DecisionTreeClassifier(**params)
                    model.fit(X_train, y_train)
                    val_pred = model.predict(X_val)
                    val_acc = manual_accuracy(y_val, val_pred)
                    results.append({**params, "val_accuracy": val_acc})
                    if val_acc > best["acc"]:
                        best = {"acc": val_acc, "params": params, "model": model}
    results_df = pd.DataFrame(results).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
    return best, results_df

## Section 7 & 8: XGBoost Classifier + Hyperparameter Tuning
Same hyperparameters and search strategy as the regression notebook
(fixed-seed random search over 60 combinations drawn from the 288-combination
grid), with `objective="binary:logistic"` and validation scored with
`manual_accuracy`.

In [7]:
XGB_GRID = {
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 5],
    "gamma": [0, 0.1],
}


def tune_xgboost(X_train, y_train, X_val, y_val, grid=XGB_GRID, n_random=60, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    keys = list(grid.keys())
    all_combos = []
    for _ in range(n_random * 3):  # oversample then dedupe
        combo = {k: grid[k][rng.randint(len(grid[k]))] for k in keys}
        all_combos.append(tuple(sorted(combo.items())))
    unique_combos = list(dict.fromkeys(all_combos))[:n_random]

    results = []
    best = {"acc": -np.inf, "params": None, "model": None}
    for combo in unique_combos:
        params = dict(combo)
        model = XGBClassifier(
            objective="binary:logistic",
            random_state=RANDOM_STATE,
            n_jobs=4,
            verbosity=0,
            eval_metric="logloss",
            **params
        )
        model.fit(X_train, y_train)
        val_pred = model.predict(X_val)
        val_acc = manual_accuracy(y_val, val_pred)
        results.append({**params, "val_accuracy": val_acc})
        if val_acc > best["acc"]:
            best = {"acc": val_acc, "params": params, "model": model}
    results_df = pd.DataFrame(results).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
    return best, results_df

## Section 10 & 11: Final Evaluation + Results Comparison

In [8]:
def run_pipeline(path, loader, dataset_name):
    sep = "=" * 70
    print(f"\n{sep}\nDATASET: {dataset_name}\n{sep}")
    df_raw, target = loader(path)
    df, insp = clean_and_inspect(df_raw, dataset_name)
    print(json.dumps(insp, indent=2))
    print("Class balance (status):")
    print(df[target].value_counts().to_dict())

    X, y = encode_features(df, target)
    (X_train, y_train), (X_val, y_val), (X_test, y_test) = sequential_split(X, y)
    print(f"Train size: {len(X_train)}  Val size: {len(X_val)}  Test size: {len(X_test)}")

    # ---- Decision Tree ----
    dt_best, dt_results = tune_decision_tree(X_train, y_train, X_val, y_val)
    dt_model = dt_best["model"]
    dt_test_pred = dt_model.predict(X_test)
    dt_test_report = manual_classification_report(y_test, dt_test_pred)
    print("\nBest Decision Tree params:", dt_best["params"])
    print("Decision Tree Validation Accuracy:", dt_best["acc"])
    print("Decision Tree Test Report:", dt_test_report)

    # ---- XGBoost ----
    xgb_best, xgb_results = tune_xgboost(X_train, y_train, X_val, y_val)
    xgb_model = xgb_best["model"]
    xgb_test_pred = xgb_model.predict(X_test)
    xgb_test_report = manual_classification_report(y_test, xgb_test_pred)
    print("\nBest XGBoost params:", xgb_best["params"])
    print("XGBoost Validation Accuracy:", xgb_best["acc"])
    print("XGBoost Test Report:", xgb_test_report)

    return {
        "dataset_name": dataset_name,
        "inspection": insp,
        "n_features": X.shape[1],
        "feature_names": list(X.columns),
        "split_sizes": {"train": len(X_train), "val": len(X_val), "test": len(X_test)},
        "dt_best_params": dt_best["params"],
        "dt_val_accuracy": dt_best["acc"],
        "dt_test_report": dt_test_report,
        "dt_top5": dt_results.head(5).to_dict(orient="records"),
        "xgb_best_params": xgb_best["params"],
        "xgb_val_accuracy": xgb_best["acc"],
        "xgb_test_report": xgb_test_report,
        "xgb_top5": xgb_results.head(5).to_dict(orient="records"),
    }

### Run on BC-BrainCancer (target: `status`)

In [9]:
results_braincancer = run_pipeline(
    "BC-BrainCancer.csv", load_brain_cancer, "BC-BrainCancer (status)"
)


DATASET: BC-BrainCancer (status)


FileNotFoundError: [Errno 2] No such file or directory: 'BC-BrainCancer.csv'

### Results Comparison Table

In [ ]:
comparison_rows = []
for model_key, label in [("dt", "Decision Tree"), ("xgb", "XGBoost")]:
    rep = results_braincancer[f"{model_key}_test_report"]
    comparison_rows.append({
        "Model": label,
        "Validation Accuracy": results_braincancer[f"{model_key}_val_accuracy"],
        "Test Accuracy": rep["accuracy"],
        "Test Precision": rep["precision"],
        "Test Recall": rep["recall"],
        "Test F1": rep["f1"],
        "TP": rep["confusion_matrix"]["TP"],
        "TN": rep["confusion_matrix"]["TN"],
        "FP": rep["confusion_matrix"]["FP"],
        "FN": rep["confusion_matrix"]["FN"],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,Model,Validation Accuracy,Test Accuracy,Test Precision,Test Recall,Test F1,TP,TN,FP,FN
0,Decision Tree,0.846154,0.714286,0.4,0.666667,0.50,2,8,3,1
1,XGBoost,0.769231,0.857143,0.6,1.000000,0.75,3,9,2,0


In [ ]:
with open("results_classification.json", "w") as f:
    json.dump(results_braincancer, f, indent=2, default=str)
print("Saved results_classification.json")

Saved results_classification.json
